# Pilot 5 on the pod

One cell. Three rising beeps = done, two low beeps = stopped (scroll up). Needs the network volume (model cache; pilot 4's recordings for a consistency check), so the pod must be in the volume's data centre (US-TX-3).

In [ ]:
# ===== Pilot 5: the pre-registered steering test + the circuit (about 60-75 minutes) =====
# Pod: a 48 GB GPU (L40S or RTX A6000), Secure Cloud, US-TX-3, with the network volume attached (at /workspace).
# The log shows progress only; findings are in the results files.
# At the three rising beeps: download /workspace/pilot5_results.tar.gz and send it to Claude.
import os, subprocess, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P5 = f"{REPO}/pilot5-value-circuit"
os.environ["HF_HOME"] = "/workspace/hf"          # model cache on the network volume (already has Qwen3-14B)

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"set -o pipefail; ({cmd}) 2>&1 | tee {log_file}"
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}")

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

run_all([
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git fetch -q origin main && git reset -q --hard origin/main && git log -1 --format='Running commit %h: %s'",
    f"cd {P5} && pip install -q -r requirements.txt",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1 "
    "&& (du -sh /workspace/acts /workspace/hf 2>/dev/null || echo 'pilot 4 recordings not found (fine: only used for a check)')",
    # A few-minute test of every stage on a small model, before the big run.
    (f"cd {P5} && python pipeline.py --quick && python tests/test_mechanics.py --quick", "/workspace/log_pilot5_quick.txt"),
    # Pre-registration: the real run only starts once both sets of predictions are in the repo.
    f"cd {P5} && test -f predictions/JASPER.md && test -f predictions/CLAUDE.md && sha256sum -c predictions/hashes.sha256 "
    "|| (echo 'Predictions are not committed yet: send them to Claude first, then run this cell again.' && exit 1)",
    # The real run (the steering test is stage 3).
    (f"cd {P5} && python pipeline.py", "/workspace/log_pilot5.txt"),
    f"cd /workspace && cp log_pilot5.txt log_pilot5_quick.txt {P5}/results/real/ && "
    f"tar czf pilot5_results.tar.gz -C {P5}/results real && ls -la /workspace/pilot5_results.tar.gz && df -h /workspace | tail -1",
])
